# Çınlar Chapter 5 Tutorial: Markov Chains

This notebook is a guided tutorial for Chapter 5, **Markov Chains**. It follows the chapter's main development:

1. Markov chains and transition matrices
2. Multi-step transition probabilities and the Chapman--Kolmogorov equations
3. Examples: Bernoulli counts, success times, independent trials, random walks, inventory, lifetimes
4. Conditional expectations and the strong Markov property
5. Visits to a fixed state, first return/first hit probabilities, total visits, and the potential matrix
6. Classification of states: recurrent/transient, null/non-null, periodic/aperiodic
7. Communicating classes, closed classes, finite-state consequences
8. Absorbing chains and ruin-type computations

The mental model for the chapter is simple:

> A Markov chain is a random system whose future depends on the past only through its present state.

The entire chapter is about making that sentence computational.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fractions import Fraction

np.set_printoptions(precision=4, suppress=True)

def matrix_power(P, n):
    """n-step transition matrix P^n."""
    return np.linalg.matrix_power(np.asarray(P, dtype=float), n)

def simulate_markov(P, start, n, rng=None):
    """Simulate X_0, ..., X_n for a finite Markov chain."""
    rng = np.random.default_rng(rng)
    P = np.asarray(P, dtype=float)
    states = [start]
    x = start
    for _ in range(n):
        x = rng.choice(len(P), p=P[x])
        states.append(x)
    return np.array(states)

def pretty(P, labels=None):
    P = np.asarray(P, dtype=float)
    if labels is None:
        labels = list(range(P.shape[0]))
    return pd.DataFrame(P, index=labels, columns=labels)

## 1. Markov chains

Let \((\Omega, \mathcal F, P)\) be a probability space. A stochastic process

$$
X = \{X_n : n \in \mathbb N\}
$$

taking values in a countable state space \(E\) is a **Markov chain** if

$$
P(X_{n+1}=j \mid X_0=i_0,\ldots,X_n=i)=P(X_{n+1}=j \mid X_n=i)
$$

for all states involved.

If the one-step transition probability does not depend on time \(n\), write

$$
P(i,j)=P(X_{n+1}=j\mid X_n=i).
$$

The matrix \(P=(P(i,j))_{i,j\in E}\) is the **transition matrix**.

### Thinking model

A Markov chain is not memoryless in the sense that the whole path has no information. Rather:

> Once the current state is known, the earlier path has no additional predictive value for the next state.

The current state is a compressed sufficient summary of the past.

## 2. Markov matrices

A matrix \(P\) is a **Markov matrix** or **stochastic matrix** if

1. \(P(i,j)\ge 0\) for all \(i,j\),
2. each row sums to one:

$$
\sum_j P(i,j)=1.
$$

Rows are "from" states; columns are "to" states.

For a row probability vector \(\pi\), the distribution after one step is

$$
\pi P.
$$

After \(n\) steps, it is

$$
\pi P^n.
$$

In [ ]:
# A two-state Markov chain, similar to Chapter 5's small finite examples.
P = np.array([
    [0.5, 0.5],
    [0.3, 0.7],
])
pretty(P, labels=[1, 2])

In [ ]:
pi0 = np.array([1/3, 2/3])
for n in range(6):
    print(f"n={n}: pi_0 P^{n} =", pi0 @ matrix_power(P, n))

## 3. Path probabilities

If \(X\) is a Markov chain with initial distribution \(\pi\), then for a finite path

$$
i_0,i_1,\ldots,i_n,
$$

the probability of observing that path is

$$
P(X_0=i_0,\ldots,X_n=i_n)
= \pi(i_0)P(i_0,i_1)P(i_1,i_2)\cdots P(i_{n-1},i_n).
$$

### Proof

Use the multiplication rule for conditional probabilities:

$$
P(A_0\cap\cdots\cap A_n)
= P(A_0)P(A_1\mid A_0)\cdots P(A_n\mid A_0,\ldots,A_{n-1}).
$$

Set \(A_k=\{X_k=i_k\}\). The Markov property replaces each long conditional probability by

$$
P(X_k=i_k\mid X_0=i_0,\ldots,X_{k-1}=i_{k-1})
= P(i_{k-1},i_k).
$$

In [ ]:
# Path probability example: start distribution pi0, path 2 -> 1 -> 2 -> 2.
# Python labels: state 1 is index 0; state 2 is index 1.
path = [1, 0, 1, 1]
prob = pi0[path[0]]
for a, b in zip(path, path[1:]):
    prob *= P[a, b]
prob

## 4. Multi-step transitions and Chapman--Kolmogorov

The \(n\)-step transition probability is

$$
P_n(i,j)=P(X_{m+n}=j\mid X_m=i).
$$

For a time-homogeneous Markov chain,

$$
P_n(i,j) = P^n(i,j).
$$

The Chapman--Kolmogorov equation says

$$
P_{m+n}(i,j)=\sum_k P_m(i,k)P_n(k,j).
$$

In matrix form:

$$
P^{m+n}=P^mP^n.
$$

### Proof idea

To go from \(i\) to \(j\) in \(m+n\) steps, condition on the intermediate state after \(m\) steps:

$$
\{X_{m+n}=j\}=\bigcup_k \{X_m=k, X_{m+n}=j\}.
$$

The events are disjoint, and the Markov property gives the product.

In [ ]:
for n in [1, 2, 3, 10]:
    print(f"P^{n} =")
    print(matrix_power(P, n))
    print()

## 5. Example: a finite three-state chain

The chapter uses small finite chains to show how path probabilities and multi-step probabilities are computed. Here is a representative three-state example.

Let \(E=\{a,b,c\}\) and

$$
P=\begin{pmatrix}
2/5 & 1/5 & 2/5\\
1/3 & 1/3 & 1/3\\
1/4 & 1/2 & 1/4
\end{pmatrix}.
$$

Then \(P^2(i,j)\) gives the probability of moving from \(i\) to \(j\) in exactly two steps.

In [ ]:
labels = ['a', 'b', 'c']
P3 = np.array([
    [2/5, 1/5, 2/5],
    [1/3, 1/3, 1/3],
    [1/4, 1/2, 1/4],
])
pretty(P3, labels)

In [ ]:
pretty(matrix_power(P3, 2), labels)

In [ ]:
# Probability of the path c -> a -> b -> c -> c -> a
path_labels = ['c', 'a', 'b', 'c', 'c', 'a']
idx = {s:i for i,s in enumerate(labels)}
path = [idx[s] for s in path_labels]
prob = 1.0  # conditioned on X_0 = c
for a,b in zip(path, path[1:]):
    prob *= P3[a,b]
prob

## 6. Example: Bernoulli success counts are Markov

Let \(Y_1,Y_2,\ldots\) be independent Bernoulli trials with success probability \(p\), and define

$$
N_n=Y_1+\cdots+Y_n.
$$

Then \(N_n\) is a Markov chain on \(\{0,1,2,\ldots\}\). From state \(i\), the chain either stays at \(i\) after a failure or moves to \(i+1\) after a success:

$$
P(i,j)=
\begin{cases}
q, & j=i,\\
p, & j=i+1,\\
0, & \text{otherwise,}
\end{cases}
\quad q=1-p.
$$

The state \(N_n\) already records all successes so far. The earlier order of successes and failures is irrelevant for predicting \(N_{n+1}\).

In [ ]:
def bernoulli_count_matrix(p, max_state=8):
    q = 1 - p
    P = np.zeros((max_state+1, max_state+1))
    for i in range(max_state+1):
        P[i, i] += q
        if i + 1 <= max_state:
            P[i, i+1] += p
        else:
            P[i, i] += p  # boundary artifact for finite truncation
    return P

PB = bernoulli_count_matrix(0.3, 8)
pretty(PB, labels=list(range(9)))

The \(m\)-step transition is binomial:

$$
P_m(i,j)=\binom{m}{j-i}p^{j-i}q^{m-(j-i)},\qquad j=i,i+1,\ldots,i+m.
$$

That is, over the next \(m\) trials, the chain gains exactly \(j-i\) successes.

In [ ]:
from math import comb
p = 0.3
q = 1-p
m = 5
i = 2
dist = {j: comb(m, j-i)*p**(j-i)*q**(m-(j-i)) for j in range(i, i+m+1)}
dist

## 7. Example: times of successes are Markov

Let \(T_k\) be the trial number of the \(k\)-th success in independent Bernoulli trials.

Then \(T_k\) is also a Markov chain. If \(T_k=i\), then the next success time is

$$
T_{k+1}=i+G,
$$

where \(G\) has the geometric distribution

$$
P(G=m)=q^{m-1}p,\qquad m=1,2,\ldots.
$$

Therefore

$$
P(i,j)=P(T_{k+1}=j\mid T_k=i)=q^{j-i-1}p,
\qquad j=i+1,i+2,\ldots.
$$

The future after a success time restarts because Bernoulli trials are independent.

In [ ]:
def success_time_transition(p, max_t=10):
    q = 1-p
    P = np.zeros((max_t+1, max_t+1))
    for i in range(max_t+1):
        for j in range(i+1, max_t+1):
            P[i,j] = (q**(j-i-1))*p
        # missing tail due to truncation stays omitted
    return P

PT = success_time_transition(0.4, 10)
pretty(PT, labels=list(range(11))).round(4)

## 8. Example: independent trials process

Let \(X_0,X_1,\ldots\) be independent random variables with the same distribution \(\pi\). Then

$$
P(X_{n+1}=j\mid X_n=i)=\pi(j).
$$

So \(X_n\) is a Markov chain with transition matrix whose every row is \(\pi\):

$$
P=\begin{pmatrix}
\pi(0)&\pi(1)&\pi(2)&\cdots\\
\pi(0)&\pi(1)&\pi(2)&\cdots\\
\pi(0)&\pi(1)&\pi(2)&\cdots\\
\vdots&\vdots&\vdots&\ddots
\end{pmatrix}.
$$

This is an extreme case: the next state is independent even of the current state.

In [ ]:
pi = np.array([0.2, 0.5, 0.3])
P_iid = np.tile(pi, (3,1))
pretty(P_iid, labels=['red','green','blue'])

In [ ]:
print("P^2 =")
print(matrix_power(P_iid, 2))
print("P^5 =")
print(matrix_power(P_iid, 5))

## 9. Example: sums of independent random variables

Let \(Y_1,Y_2,\ldots\) be independent identically distributed integer-valued random variables, and define

$$
X_n=Y_1+\cdots+Y_n.
$$

Then \(X_n\) is a Markov chain because

$$
X_{n+1}=X_n+Y_{n+1}.
$$

If \(p_k=P(Y_1=k)\), then

$$
P(i,j)=P(Y_{n+1}=j-i)=p_{j-i}.
$$

This is a random walk on the integers or nonnegative integers, depending on the support of \(Y_n\).

In [ ]:
# Random walk with increments -1, 0, +1 on a finite window {-4,...,4}.
states = np.arange(-4, 5)
prob_inc = {-1: 0.25, 0: 0.25, 1: 0.50}
Prw = np.zeros((len(states), len(states)))
for a,i in enumerate(states):
    for inc, prob in prob_inc.items():
        j = i + inc
        if j in states:
            Prw[a, np.where(states == j)[0][0]] += prob
        else:
            # reflecting boundary just for finite visualization
            Prw[a, a] += prob
pretty(Prw, states)

In [ ]:
path = simulate_markov(Prw, start=np.where(states==0)[0][0], n=100, rng=7)
values = states[path]
plt.figure(figsize=(9,3))
plt.plot(values)
plt.axhline(0, linewidth=1)
plt.title("Simulated random walk path")
plt.xlabel("n")
plt.ylabel("X_n")
plt.show()

## 10. Example: random walk modulo 5

Let \(Y_n\in\{0,1,2,3,4\}\) be i.i.d. with probabilities \(p_0,\ldots,p_4\), and define

$$
X_{n+1}=X_n+Y_{n+1}\pmod 5.
$$

Then \(X_n\) is a Markov chain on \(\{0,1,2,3,4\}\), with

$$
P(i,j)=p_{(j-i)\bmod 5}.
$$

This gives a circulant transition matrix. Each row is a shifted copy of the previous row. If each column also sums to one, the matrix is **doubly stochastic**.

In [ ]:
p_mod = np.array([0.1, 0.2, 0.3, 0.25, 0.15])
Pmod = np.zeros((5,5))
for i in range(5):
    for j in range(5):
        Pmod[i,j] = p_mod[(j-i) % 5]
pretty(Pmod)

In [ ]:
print("Row sums:", Pmod.sum(axis=1))
print("Column sums:", Pmod.sum(axis=0))

## 11. Example: inventory chain

In an inventory problem, let \(X_n\) be stock level just before demand during period \([t_n,t_{n+1})\). Suppose demand during the period is \(Z_{n+1}\), and an order-up-to policy is used:

- if stock after demand is \(\le s\), replenish to \(S\),
- otherwise keep the remaining stock.

Then

$$
X_{n+1}=\begin{cases}
S-Z_{n+1}, & X_n-Z_{n+1}\le s,\\
X_n-Z_{n+1}, & X_n-Z_{n+1}>s.
\end{cases}
$$

If demands \(Z_n\) are independent and identically distributed, then \(X_n\) is Markov.

The current stock is enough to determine the distribution of the next stock.

In [ ]:
def inventory_transition(s=1, S=5, demand_probs=None):
    # States are 0..S. Demand is clipped if it exceeds available stock for this toy example.
    if demand_probs is None:
        demand_probs = {0:0.1, 1:0.3, 2:0.4, 3:0.2}
    states = np.arange(S+1)
    P = np.zeros((S+1, S+1))
    for x in states:
        for z, prob in demand_probs.items():
            after = max(x-z, 0)
            nxt = S if after <= s else after
            P[x, nxt] += prob
    return P

Pinv = inventory_transition(s=1, S=5)
pretty(Pinv)

## 12. Example: replacement/lifetime chain

Suppose a component is replaced immediately on failure by an identical component. Let \(X_n\) be the remaining lifetime at time \(n\). If lifetimes of successive components are i.i.d. with distribution \(p_1,p_2,\ldots\), then the remaining lifetime evolves as follows:

- if \(X_n=i\ge 1\), then \(X_{n+1}=i-1\),
- if \(X_n=0\), a new lifetime is sampled.

So

$$
P(i,j)=
\begin{cases}
1, & i\ge 1,\ j=i-1,\\
p_{j+1}, & i=0,\\
0, & \text{otherwise.}
\end{cases}
$$

Again, the current remaining lifetime summarizes all relevant past information.

In [ ]:
life_probs = np.array([0.2, 0.5, 0.3])  # lifetimes 1,2,3
max_rem = 3
Plife = np.zeros((max_rem+1, max_rem+1))
Plife[0, 0:3] = life_probs  # new lifetime L gives remaining L-1 after one time unit in this indexing
for i in range(1, max_rem+1):
    Plife[i, i-1] = 1
pretty(Plife, labels=list(range(max_rem+1)))

## 13. Conditional expectations for Markov chains

The chapter repeatedly uses this principle:

If \(Y\) is a bounded function of the future \(X_{n+1},X_{n+2},\ldots\), then

$$
E[Y\mid X_0,\ldots,X_n] = E[Y\mid X_n].
$$

For example, if

$$
Y=f(X_{n+1},\ldots,X_{n+k}),
$$

then there is a function \(g\) such that

$$
E[f(X_{n+1},\ldots,X_{n+k})\mid X_0,\ldots,X_n] = g(X_n).
$$

### Computational form

For one future step,

$$
E[f(X_{n+1})\mid X_n=i]=\sum_j f(j)P(i,j).
$$

In vector notation, if \(f\) is a column vector, this is \((Pf)(i)\).

In [ ]:
f = np.array([10, 20, 40])
print("P3 f = expected next reward given current state")
print(P3 @ f)

## 14. Stopping times and the strong Markov property

A random time \(T\) is a **stopping time** if whether \(T=n\) is determined by the observed history up to time \(n\):

$$
\{T=n\}\in \sigma(X_0,\ldots,X_n).
$$

Examples:

- first time a chain hits a state,
- first return time to a state,
- first time inventory drops below a threshold,
- first time a random walk reaches a boundary.

The **strong Markov property** says that, conditional on \(X_T\), the future after a stopping time behaves like a fresh Markov chain started from \(X_T\):

$$
P(X_{T+m}=j\mid X_T=i,\ T<\infty)=P^m(i,j).
$$

This is the ordinary Markov property with the deterministic time \(n\) replaced by a random observable time \(T\).

In [ ]:
# Strong Markov property demonstration by simulation:
# two-state chain; T = first hit of state 1. After T, next-step distribution should be row P[1].
rng = np.random.default_rng(123)
post_next = []
for _ in range(20000):
    xs = simulate_markov(P, start=0, n=20, rng=rng)
    hits = np.where(xs == 1)[0]
    if len(hits) > 0 and hits[0] < len(xs)-1:
        T = hits[0]
        post_next.append(xs[T+1])
post_next = np.array(post_next)
emp = np.bincount(post_next, minlength=2) / len(post_next)
print("Empirical distribution of X_{T+1} after first hit of state 1:", emp)
print("Theoretical P[1,:]:", P[1])

## 15. Visits to a fixed state

Fix a state \(j\). Define

$$
N_j=\sum_{n\ge 0} 1_{\{X_n=j\}},
$$

the total number of visits to \(j\).

Define the first hitting time of \(j\):

$$
T_j=\inf\{n\ge 1:X_n=j\}.
$$

The first-hit probabilities are

$$
F(i,j)=P_i(T_j<\infty).
$$

The \(k\)-step first-hit probabilities are

$$
F_k(i,j)=P_i(T_j=k).
$$

They satisfy the recursion

$$
F_1(i,j)=P(i,j),
$$

and for \(k\ge 2\),

$$
F_k(i,j)=\sum_{b\ne j}P(i,b)F_{k-1}(b,j).
$$

The condition \(b\ne j\) enforces that this is the **first** visit to \(j\).

In [ ]:
def first_hit_probs(P, target, max_k):
    P = np.asarray(P, dtype=float)
    n = P.shape[0]
    F = np.zeros((max_k+1, n))
    # F[k, i] = P_i(T_target = k), k >= 1
    F[1, :] = P[:, target]
    F[1, target] = P[target, target]  # first return when starting at target occurs at step 1 if self-loop
    for k in range(2, max_k+1):
        for i in range(n):
            F[k, i] = sum(P[i,b] * F[k-1,b] for b in range(n) if b != target)
    return F

F = first_hit_probs(P3, target=2, max_k=10)
pd.DataFrame(F[1:].T, index=labels, columns=[f"k={k}" for k in range(1,11)]).round(4)

The probability of ever hitting \(j\) is

$$
F(i,j)=\sum_{k\ge 1}F_k(i,j).
$$

For computations, truncate the sum or solve linear equations.

In [ ]:
# Approximate ever-hit probabilities by summing first-hit probabilities up to k=50.
F50 = first_hit_probs(P3, target=2, max_k=50)
F_approx = F50[1:].sum(axis=0)
pd.Series(F_approx, index=labels, name="approx P_i(T_c < infinity)")

## 16. Total number of visits and geometric structure

Starting from state \(j\), after each visit to \(j\), the chain either eventually returns to \(j\) or never returns. Let

$$
F(j,j)=P_j(T_j<\infty).
$$

Then

$$
P_j(N_j=m)=F(j,j)^{m-1}\bigl(1-F(j,j)\bigr),\qquad m=1,2,\ldots,
$$

if \(F(j,j)<1\). If \(F(j,j)=1\), then \(N_j=\infty\) almost surely.

Thus:

- \(j\) is **recurrent** if \(F(j,j)=1\),
- \(j\) is **transient** if \(F(j,j)<1\).

## 17. Potential matrix

The expected number of visits to \(j\), starting from \(i\), is

$$
R(i,j)=E_i[N_j].
$$

Since

$$
N_j=\sum_{n\ge 0}1_{\{X_n=j\}},
$$

we get

$$
R(i,j)=\sum_{n\ge 0}P^n(i,j).
$$

In matrix form,

$$
R=I+P+P^2+\cdots.
$$

If the series converges, then

$$
R(I-P)=I,
$$

so

$$
R=(I-P)^{-1}.
$$

This is the **potential matrix**.

In [ ]:
# Potential matrix for a transient substochastic chain.
# Think of states 0,1 as non-absorbing and leakage to an absorbing outside state.
Q = np.array([
    [0.2, 0.3],
    [0.4, 0.1],
])
R = np.linalg.inv(np.eye(2) - Q)
print(R)
print("Check R(I-Q):")
print(R @ (np.eye(2)-Q))

## 18. Recurrent, transient, null, non-null, periodic, aperiodic

A state \(j\) is:

### Recurrent

$$
P_j(T_j<\infty)=1.
$$

Starting from \(j\), the chain returns to \(j\) eventually with probability one.

### Transient

$$
P_j(T_j<\infty)<1.
$$

Starting from \(j\), there is positive probability that the chain never returns.

### Null recurrent vs non-null recurrent

If \(j\) is recurrent, define the expected return time

$$
E_j[T_j].
$$

- recurrent and \(E_j[T_j]=\infty\): **null recurrent**
- recurrent and \(E_j[T_j]<\infty\): **non-null recurrent** or **positive recurrent**

### Periodic vs aperiodic

The period of \(j\) is

$$
d(j)=\gcd\{n\ge 1:P^n(j,j)>0\}.
$$

- if \(d(j)\ge 2\), \(j\) is periodic,
- if \(d(j)=1\), \(j\) is aperiodic.

In [ ]:
from math import gcd
from functools import reduce

def period(P, state, max_n=30, tol=1e-12):
    P = np.asarray(P, dtype=float)
    ns = []
    M = np.eye(P.shape[0])
    for n in range(1, max_n+1):
        M = M @ P
        if M[state, state] > tol:
            ns.append(n)
    if not ns:
        return None, ns
    return reduce(gcd, ns), ns

# Period-2 chain: 0 <-> 1
P_period2 = np.array([[0,1],[1,0]])
print(period(P_period2, 0, 10))

# Aperiodic chain with self-loops
print(period(P, 0, 10))

## 19. Communication and classes

State \(i\) **leads to** state \(j\), written \(i\to j\), if

$$
P^n(i,j)>0
$$

for some \(n\ge 0\).

States \(i\) and \(j\) **communicate**, written \(i\leftrightarrow j\), if

$$
i\to j \quad\text{and}\quad j\to i.
$$

Communication is an equivalence relation. Its equivalence classes are called **communicating classes**.

A class \(C\) is **closed** if once the chain enters \(C\), it cannot leave:

$$
P(i,j)=0 \quad\text{for all } i\in C,\ j\notin C.
$$

A finite closed irreducible class is recurrent.

In [ ]:
def reachability(P, max_n=None, tol=1e-12):
    P = np.asarray(P, dtype=float)
    n = P.shape[0]
    if max_n is None:
        max_n = n
    R = np.eye(n, dtype=bool)
    M = np.eye(n)
    for _ in range(1, max_n+1):
        M = M @ P
        R |= (M > tol)
    return R

def communicating_classes(P):
    R = reachability(P)
    n = len(P)
    unused = set(range(n))
    classes = []
    while unused:
        i = unused.pop()
        C = {j for j in range(n) if R[i,j] and R[j,i]}
        classes.append(sorted(C))
        unused -= C
    return classes

P_classes = np.array([
    [0.5,0.5,0,0],
    [0.5,0.5,0,0],
    [0.2,0,0.3,0.5],
    [0,0,0.4,0.6],
])
pretty(P_classes)

In [ ]:
print("Reachability matrix:")
print(reachability(P_classes).astype(int))
print("Communicating classes:", communicating_classes(P_classes))

In the example above:

- states \(0\) and \(1\) form a closed class,
- states \(2\) and \(3\) communicate with each other, but the class is not closed because state \(2\) can move to state \(0\).

The closed finite class is recurrent. The non-closed class is transient.

## 20. Absorbing states and absorbing chains

A state \(a\) is **absorbing** if

$$
P(a,a)=1.
$$

For an absorbing chain, after reordering states, the transition matrix often has block form

$$
P=\begin{pmatrix}
Q & R\\
0 & I
\end{pmatrix},
$$

where \(Q\) describes transient-to-transient moves and \(R\) describes transient-to-absorbing moves.

The fundamental matrix is

$$
N=(I-Q)^{-1}=I+Q+Q^2+\cdots.
$$

Its entry \(N(i,j)\\) is the expected number of visits to transient state \(j\), starting from transient state \(i\), before absorption.

The absorption probabilities are

$$
B=NR.
$$

In [ ]:
# Gambler's ruin: states 0 and N are absorbing.
# From i, move to i+1 with p and i-1 with q.
def gambler_ruin_matrix(N=5, p=0.5):
    q = 1-p
    P = np.zeros((N+1, N+1))
    P[0,0] = 1
    P[N,N] = 1
    for i in range(1,N):
        P[i,i-1] = q
        P[i,i+1] = p
    return P

Pgr = gambler_ruin_matrix(N=5, p=0.55)
pretty(Pgr)

In [ ]:
# Absorption probabilities and expected time to absorption.
Ncap = 5
transient = list(range(1,Ncap))
absorbing = [0,Ncap]
Q = Pgr[np.ix_(transient, transient)]
Rblock = Pgr[np.ix_(transient, absorbing)]
Fund = np.linalg.inv(np.eye(len(Q)) - Q)
B = Fund @ Rblock
expected_time = Fund @ np.ones(len(Q))

print("Fundamental matrix N=(I-Q)^-1:")
print(Fund.round(4))
print("Absorption probabilities into 0 and N:")
print(pd.DataFrame(B, index=transient, columns=absorbing).round(4))
print("Expected time to absorption:")
print(pd.Series(expected_time, index=transient).round(4))

## 21. Classification examples

### Example A: irreducible finite chain

If a finite Markov chain is irreducible, all states communicate. Then all states are recurrent and positive recurrent.

The two-state chain

$$
P=\begin{pmatrix}0.5&0.5\\0.3&0.7\end{pmatrix}
$$

is irreducible and aperiodic.

### Example B: deterministic alternation

$$
P=\begin{pmatrix}0&1\\1&0\end{pmatrix}
$$

is irreducible and recurrent, but periodic with period 2.

### Example C: absorbing state

$$
P=\begin{pmatrix}1&0\\0.4&0.6\end{pmatrix}
$$

State 0 is absorbing and recurrent. State 1 is transient because it may move to 0 and then never return.

In [ ]:
examples = {
    "irreducible_aperiodic": P,
    "period_2": P_period2,
    "absorbing": np.array([[1,0],[0.4,0.6]])
}
for name, M in examples.items():
    print("\n", name)
    print(pretty(M))
    for s in range(M.shape[0]):
        print("state", s, "period", period(M, s, 12)[0], "return steps", period(M, s, 12)[1])

## 22. Stationary distributions

Although the chapter's first goal is classification, transition matrices naturally lead to **stationary distributions**. A row probability vector \(\pi\) is stationary if

$$
\pi P = \pi.
$$

Interpretation:

> If the chain starts with distribution \(\pi\), then after one step it still has distribution \(\pi\).

For finite irreducible positive recurrent chains, the stationary distribution exists and is unique.

In [ ]:
def stationary_distribution(P):
    P = np.asarray(P, dtype=float)
    n = P.shape[0]
    A = (P.T - np.eye(n))
    A[-1] = np.ones(n)
    b = np.zeros(n)
    b[-1] = 1
    return np.linalg.solve(A, b)

for name, M in {"two-state": P, "mod-5": Pmod, "three-state": P3}.items():
    pi = stationary_distribution(M)
    print(name, pi, "check", pi @ M)

## 23. Long-run simulation

For many irreducible, aperiodic finite chains, the empirical fraction of time spent in state \(j\) converges to the stationary probability \(\pi(j)\).

This connects the chapter's visit-count point of view to long-run averages.

In [ ]:
xs = simulate_markov(P3, start=0, n=100_000, rng=42)
empirical = np.bincount(xs, minlength=3) / len(xs)
theory = stationary_distribution(P3)
pd.DataFrame({"empirical": empirical, "stationary": theory}, index=labels)

## 24. Proof capsule: why recurrence is class-wide

If \(i\leftrightarrow j\), then recurrence or transience is shared by \(i\) and \(j\).

### Intuition

If the chain can go from \(i\) to \(j\), and from \(j\) to \(i\), then repeated returns to one state force repeated opportunities to visit the other.

### Proof sketch

Suppose \(i\to j\) and \(j\to i\). Then for some \(m,n\),

$$
P^m(i,j)>0,\qquad P^n(j,i)>0.
$$

If \(i\) is recurrent, starting at \(i\) the chain returns to \(i\) infinitely often almost surely. Each excursion beginning at \(i\) has a positive probability of reaching \(j\) in \(m\) steps. Repeated independent-ish opportunities, formalized using the strong Markov property, imply that \(j\) is hit almost surely. Once at \(j\), the same argument gives return to \(j\) almost surely.

Therefore recurrence/transience is a property of communicating classes, not merely individual states.

## 25. Proof capsule: finite closed classes are recurrent

Let \(C\) be a finite closed communicating class.

Because \(C\) is closed, once the chain enters \(C\), it stays there forever. Since \(C\) is finite, the chain must visit some state in \(C\) infinitely often. Since all states in \(C\) communicate, if one state is visited infinitely often, every state in the class must be revisited infinitely often. Hence all states in \(C\) are recurrent.

This is one of the chapter's most important structural consequences:

> In a finite Markov chain, every closed communicating class is recurrent, and states outside closed classes are transient.

## 26. Exercises for mastery

Use these to test the concepts from the chapter.

### Exercise 1: path probability

For

$$
P=\begin{pmatrix}0.5&0.5\\0.3&0.7\end{pmatrix},
$$

compute

$$
P(X_0=1,X_1=2,X_2=2,X_3=1)
$$

when \(\pi=(1/3,2/3)\).

### Exercise 2: two-step transition

Compute \(P^2\) by hand and by Python. Interpret \(P^2(1,2)\).

### Exercise 3: first-hit probability

For the three-state chain above, compute the probability of hitting \(c\) before time 5 when starting from \(a\).

### Exercise 4: period

Find the period of each state in

$$
P=\begin{pmatrix}0&1&0\\0&0&1\\1&0&0\end{pmatrix}.
$$

### Exercise 5: absorbing chain

For gambler's ruin with states \(0,1,2,3,4,5\), \(p=0.55\), find the probability of absorption at 5 starting from state 2.

In [ ]:
# Exercise answer checks

# 1
prob_ex1 = pi0[0] * P[0,1] * P[1,1] * P[1,0]
print("Exercise 1:", prob_ex1)

# 2
print("Exercise 2 P^2:")
print(matrix_power(P, 2))

# 3
F4 = first_hit_probs(P3, target=2, max_k=4)
print("Exercise 3 P_a(T_c <= 4):", F4[1:, idx['a']].sum())

# 4
P_cycle3 = np.array([[0,1,0],[0,0,1],[1,0,0]])
print("Exercise 4 periods:")
for s in range(3):
    print(s, period(P_cycle3, s, 12))

# 5
print("Exercise 5 absorption probability at 5 from state 2:", B[transient.index(2), absorbing.index(5)])

## 27. Chapter map

The chapter's ideas can be remembered as a chain of reductions:

1. **Markov property**: the present summarizes the past.
2. **Transition matrix**: one-step dynamics become linear algebra.
3. **Matrix powers**: multi-step dynamics are powers of \(P\).
4. **Strong Markov property**: restarting works at random observable times.
5. **Visits and first returns**: recurrence/transience are about whether visits happen finitely or infinitely often.
6. **Potential matrix**: expected visit counts are \(I+P+P^2+\cdots\).
7. **Classification**: the long-term behavior is organized by communicating classes.

The practical recipe is:

- draw the transition graph,
- find communicating classes,
- identify closed classes,
- classify finite closed classes as recurrent,
- classify states that can escape to closed classes as transient,
- use matrix powers or linear equations for exact probabilities.